In [1]:
import sys

print("Python:", sys.version)
print("Ambiente:", sys.executable)

Python: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
Ambiente: c:\Users\Dell\Documents\rocketseat\ai-customer-intelligence\.venv\Scripts\python.exe


In [4]:
import pandas as pd

print("Versão do Pandas:", pd.__version__)

Versão do Pandas: 3.1.1


In [3]:
%pip install pandas

  Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   --------- ------------------------------ 2.6/10.9 MB 24.4 MB/s eta 0:00:01
   ---------------------------------------  10.7/10.9 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------- 10.9/10.9 MB 32.9 MB/s  0:00:00
Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl (12.7 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   -


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from pathlib import Path

caminho = Path("../data/raw/orders.csv")

pedidos = pd.read_csv(caminho)

print("Linhas:", pedidos.shape[0])
print("Colunas:", pedidos.shape[1])

pedidos.head()

Linhas: 368999
Colunas: 29


,order_id,store_id,channel_id,payment_order_id,delivery_order_id,order_status,order_amount,order_delivery_fee,order_delivery_cost,order_created_hour,...,order_moment_delivering,order_moment_delivered,order_moment_finished,order_metric_collected_time,order_metric_paused_time,order_metric_production_time,order_metric_walking_time,order_metric_expediton_speed_time,order_metric_transit_time,order_metric_cycle_time
0,68405119,3512,5,68405119,68405119,CANCELED,62.7,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,68405123,3512,5,68405123,68405123,CANCELED,62.7,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,68405206,3512,5,68405206,68405206,CANCELED,115.5,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,68405465,3401,5,68405465,68405465,CANCELED,55.9,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,68406064,3401,5,68406064,68406064,CANCELED,37.9,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
pedidos.info()

<class 'pandas.DataFrame'>
RangeIndex: 368999 entries, 0 to 368998
Data columns (total 29 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   order_id                           368999 non-null  int64  
 1   store_id                           368999 non-null  int64  
 2   channel_id                         368999 non-null  int64  
 3   payment_order_id                   368999 non-null  int64  
 4   delivery_order_id                  368999 non-null  int64  
 5   order_status                       368999 non-null  str    
 6   order_amount                       368999 non-null  float64
 7   order_delivery_fee                 368999 non-null  float64
 8   order_delivery_cost                361794 non-null  float64
 9   order_created_hour                 368999 non-null  int64  
 10  order_created_minute               368999 non-null  int64  
 11  order_created_day                  368999 non-null

In [7]:
pedidos["order_status"].value_counts()

order_status
FINISHED    352020
CANCELED     16979
Name: count, dtype: int64

In [8]:
total_pedidos = len(pedidos)

pedidos_cancelados = (
    pedidos["order_status"] == "CANCELED"
).sum()

taxa_cancelamento = (
    pedidos_cancelados / total_pedidos
) * 100

print(f"Total de pedidos: {total_pedidos:,}")
print(f"Pedidos cancelados: {pedidos_cancelados:,}")
print(f"Taxa de cancelamento: {taxa_cancelamento:.2f}%")

Total de pedidos: 368,999
Pedidos cancelados: 16,979
Taxa de cancelamento: 4.60%


In [9]:
pedidos.groupby("order_status")[
    ["order_moment_delivered", "order_metric_cycle_time"]
].count()

,order_moment_delivered,order_metric_cycle_time
order_status,,
CANCELED,598,1723
FINISHED,19003,351657


In [10]:
colunas_tempo = [
    "order_moment_created",
    "order_moment_accepted",
    "order_moment_ready",
    "order_moment_collected",
    "order_moment_in_expedition",
    "order_moment_delivering",
    "order_moment_delivered",
    "order_moment_finished"
]

pedidos.groupby("order_status")[colunas_tempo].count().T

order_status,CANCELED,FINISHED
order_moment_created,16979,352020
order_moment_accepted,8886,350652
order_moment_ready,2916,340977
order_moment_collected,1849,324256
order_moment_in_expedition,1671,299899
order_moment_delivering,1317,342366
order_moment_delivered,598,19003
order_moment_finished,1383,352017


In [11]:
pedidos_finalizados = pedidos[
    pedidos["order_status"] == "FINISHED"
]

pedidos_finalizados["order_metric_cycle_time"].describe()

count    351657.000000
mean        155.843596
std        1225.229132
min           0.370000
25%          32.870000
50%          42.180000
75%          55.550000
max      154277.320000
Name: order_metric_cycle_time, dtype: float64

In [12]:
tempos = pedidos_finalizados["order_metric_cycle_time"]

q1 = tempos.quantile(0.25)
q3 = tempos.quantile(0.75)

iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers = tempos[
    (tempos < limite_inferior) |
    (tempos > limite_superior)
]

print(f"Limite inferior: {limite_inferior:.2f}")
print(f"Limite superior: {limite_superior:.2f}")
print(f"Quantidade de outliers: {len(outliers):,}")
print(f"Percentual de outliers: {len(outliers) / tempos.count() * 100:.2f}%")

Limite inferior: -1.15
Limite superior: 89.57
Quantidade de outliers: 31,635
Percentual de outliers: 9.00%


In [13]:
tempos_sem_outliers = tempos[
    (tempos >= limite_inferior) &
    (tempos <= limite_superior)
]

print(f"Média original: {tempos.mean():.2f}")
print(f"Mediana original: {tempos.median():.2f}")

print(f"Média sem outliers: {tempos_sem_outliers.mean():.2f}")
print(f"Mediana sem outliers: {tempos_sem_outliers.median():.2f}")

Média original: 155.84
Mediana original: 42.18
Média sem outliers: 42.45
Mediana sem outliers: 40.47


### Análise dos valores atípicos no tempo de ciclo

Foram analisados 351.657 registros válidos de pedidos finalizados.

O método IQR identificou 31.635 valores atípicos (9,00%), com limite superior de 89,57.

A média original foi de 155,84, enquanto a mediana foi de 42,18. Após a exclusão temporária dos outliers, a média caiu para 42,45 e a mediana para 40,47.

Essa diferença indica forte influência dos valores extremos sobre a média.

**Decisão analítica:** preservar os registros originais, utilizar a mediana como referência do comportamento típico e investigar separadamente os valores extremos.

A unidade de medida e o significado operacional do tempo de ciclo ainda precisam ser confirmados.


In [14]:
tempos = pedidos_finalizados["order_metric_cycle_time"]

print("Tempos negativos:", (tempos < 0).sum())
print("Tempos zerados:", (tempos == 0).sum())

print("\nValores acima dos limites:")

for limite in [60, 120, 240, 1440]:
    quantidade = (tempos > limite).sum()
    percentual = quantidade / tempos.count() * 100

    print(
        f"Acima de {limite}: "
        f"{quantidade:,} ({percentual:.2f}%)"
    )

Tempos negativos: 0
Tempos zerados: 0

Valores acima dos limites:
Acima de 60: 71,150 (20.23%)
Acima de 120: 24,299 (6.91%)
Acima de 240: 17,281 (4.91%)
Acima de 1440: 7,469 (2.12%)


In [15]:
amostra = pedidos_finalizados[
    [
        "order_id",
        "order_moment_created",
        "order_moment_finished",
        "order_metric_cycle_time"
    ]
].dropna().head(10).copy()

inicio = pd.to_datetime(amostra["order_moment_created"])
fim = pd.to_datetime(amostra["order_moment_finished"])

amostra["diferenca_minutos"] = (
    (fim - inicio).dt.total_seconds() / 60
)

amostra["diferenca_horas"] = (
    amostra["diferenca_minutos"] / 60
)

amostra

C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\2654485319.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fim = pd.to_datetime(amostra["order_moment_finished"])


,order_id,order_moment_created,order_moment_finished,order_metric_cycle_time,diferenca_minutos,diferenca_horas
8,68410055,1/1/2021 2:32:51 AM,1/2/2021 6:57:34 PM,2424.72,2424.716667,40.411944
17,68412721,1/1/2021 2:12:11 PM,1/1/2021 4:12:36 PM,120.42,120.416667,2.006944
18,68413340,1/1/2021 2:14:51 PM,1/1/2021 3:31:54 PM,77.05,77.050000,1.284167
19,68414018,1/1/2021 2:17:31 PM,1/1/2021 3:09:20 PM,51.82,51.816667,0.863611
20,68414309,1/1/2021 2:21:02 PM,1/1/2021 2:50:20 PM,29.30,29.300000,0.488333
21,68414512,1/1/2021 2:24:01 PM,1/1/2021 3:49:47 PM,85.77,85.766667,1.429444
22,68414563,1/1/2021 2:24:26 PM,1/1/2021 3:30:56 PM,66.50,66.500000,1.108333
24,68415103,1/1/2021 2:31:33 PM,1/1/2021 3:38:55 PM,67.37,67.366667,1.122778
25,68415140,1/1/2021 2:31:52 PM,1/1/2021 4:06:40 PM,94.82,94.800000,1.580000
26,68415344,1/1/2021 2:33:32 PM,1/1/2021 4:18:52 PM,105.35,105.333333,1.755556


In [16]:
colunas_investigacao = [
    "order_id",
    "order_status",
    "order_moment_created",
    "order_moment_accepted",
    "order_moment_ready",
    "order_moment_collected",
    "order_moment_delivering",
    "order_moment_delivered",
    "order_moment_finished",
    "order_metric_cycle_time"
]

pedidos.loc[
    pedidos["order_id"] == 68410055,
    colunas_investigacao
].T

,8
order_id,68410055
order_status,FINISHED
order_moment_created,1/1/2021 2:32:51 AM
order_moment_accepted,1/1/2021 2:33:00 AM
order_moment_ready,1/2/2021 6:24:06 PM
order_moment_collected,1/2/2021 6:30:44 PM
order_moment_delivering,1/2/2021 6:35:49 PM
order_moment_delivered,NaN
order_moment_finished,1/2/2021 6:57:34 PM
order_metric_cycle_time,2424.72


In [17]:
validacao = pedidos_finalizados[
    [
        "order_moment_created",
        "order_moment_finished",
        "order_metric_cycle_time"
    ]
].dropna().copy()

inicio = pd.to_datetime(validacao["order_moment_created"])
fim = pd.to_datetime(validacao["order_moment_finished"])

validacao["tempo_calculado"] = (
    (fim - inicio).dt.total_seconds() / 60
)

validacao["diferenca_absoluta"] = (
    validacao["order_metric_cycle_time"]
    - validacao["tempo_calculado"]
).abs()

print("Registros comparados:", len(validacao))
print(
    "Diferença média absoluta:",
    round(validacao["diferenca_absoluta"].mean(), 4)
)
print(
    "Registros com diferença acima de 1 minuto:",
    (validacao["diferenca_absoluta"] > 1).sum()
)

C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\2927043599.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fim = pd.to_datetime(validacao["order_moment_finished"])


Registros comparados: 351656
Diferença média absoluta: 21.4251
Registros com diferença acima de 1 minuto: 18751


In [18]:
print("Distribuição das diferenças:")
print(validacao["diferenca_absoluta"].describe())

print("\nMaiores divergências:")
print(
    validacao.nlargest(
        5, "diferenca_absoluta"
    )[
        [
            "order_metric_cycle_time",
            "tempo_calculado",
            "diferenca_absoluta"
        ]
    ]
)

Distribuição das diferenças:
count    351656.000000
mean         21.425131
std         608.924784
min           0.000000
25%           0.003333
50%           0.003333
75%           0.013333
max      151496.583333
Name: diferenca_absoluta, dtype: float64

Maiores divergências:
       order_metric_cycle_time  tempo_calculado  diferenca_absoluta
45508                   140.55    151637.133333       151496.583333
98068                  1307.38    121533.666667       120226.286667
98300                  1260.42    121486.333333       120225.913333
98624                  1104.42    121330.150000       120225.730000
59429                    32.68     83118.083333        83085.403333


In [19]:
colunas_investigacao = [
    "order_id",
    "order_status",
    "order_moment_created",
    "order_moment_accepted",
    "order_moment_ready",
    "order_moment_collected",
    "order_moment_delivering",
    "order_moment_delivered",
    "order_moment_finished",
    "order_metric_cycle_time"
]

pedidos.loc[45508, colunas_investigacao]

order_id                               72615663
order_status                           FINISHED
order_moment_created       1/20/2021 6:13:08 PM
order_moment_accepted                       NaN
order_moment_ready                          NaN
order_moment_collected                      NaN
order_moment_delivering    1/20/2021 8:03:14 PM
order_moment_delivered     1/20/2021 8:33:41 PM
order_moment_finished       5/6/2021 1:30:16 AM
order_metric_cycle_time                  140.55
Name: 45508, dtype: object

In [20]:
validacao_entrega = pedidos_finalizados[
    [
        "order_moment_created",
        "order_moment_delivered",
        "order_metric_cycle_time"
    ]
].dropna().copy()

criacao = pd.to_datetime(
    validacao_entrega["order_moment_created"]
)

entrega = pd.to_datetime(
    validacao_entrega["order_moment_delivered"]
)

validacao_entrega["tempo_ate_entrega"] = (
    (entrega - criacao).dt.total_seconds() / 60
)

validacao_entrega["diferenca_absoluta"] = (
    validacao_entrega["order_metric_cycle_time"]
    - validacao_entrega["tempo_ate_entrega"]
).abs()

print("Registros comparados:", len(validacao_entrega))

print(
    "Diferença mediana:",
    round(validacao_entrega["diferenca_absoluta"].median(), 4)
)

print(
    "Registros com diferença acima de 1 minuto:",
    (validacao_entrega["diferenca_absoluta"] > 1).sum()
)

C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\3445740464.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  criacao = pd.to_datetime(
C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\3445740464.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  entrega = pd.to_datetime(


Registros comparados: 19003
Diferença mediana: 0.0033
Registros com diferença acima de 1 minuto: 0


### Validação do tempo de ciclo dos pedidos

Foi realizada uma comparação entre o indicador `order_metric_cycle_time`
e os intervalos calculados a partir dos registros de data e hora.

**Comparação entre criação e finalização:**
- Registros analisados: 351.656
- Diferença média absoluta: 21,43 minutos
- Diferença mediana absoluta: aproximadamente 0,0033 minuto
- Registros com divergência superior a 1 minuto: 18.751 (5,33%)

**Comparação entre criação e entrega:**
- Registros analisados: 19.003
- Diferença mediana absoluta: aproximadamente 0,0033 minuto
- Registros com divergência superior a 1 minuto: 0

**Conclusões:**

1. O indicador `order_metric_cycle_time` utiliza minutos.
2. Nos 19.003 pedidos finalizados com data de entrega disponível,
   o indicador coincide com o intervalo entre criação e entrega,
   considerando uma tolerância de 1 minuto.
3. A data de finalização nem sempre representa o momento da entrega.
4. A regra de cálculo para pedidos sem data de entrega ainda precisa
   ser investigada.
5. Os registros originais serão preservados, evitando correções
   ou exclusões sem justificativa.

**Implicação para o projeto:**

O AI Customer Intelligence deverá considerar a disponibilidade e
a consistência dos eventos antes de utilizar indicadores temporais
para avaliar o desempenho operacional dos estabelecimentos.

In [21]:
pedidos_extremos = pedidos_finalizados[
    pedidos_finalizados["order_metric_cycle_time"] > 1440
].copy()

pedidos_extremos["tempo_ate_pronto"] = (
    pd.to_datetime(pedidos_extremos["order_moment_ready"])
    - pd.to_datetime(pedidos_extremos["order_moment_accepted"])
).dt.total_seconds() / 60

print("Quantidade de pedidos extremos:", len(pedidos_extremos))

print("\nTempo entre aceitação e pedido pronto:")
print(pedidos_extremos["tempo_ate_pronto"].describe())

C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\4110037173.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(pedidos_extremos["order_moment_ready"])


Quantidade de pedidos extremos: 7469

Tempo entre aceitação e pedido pronto:
count     6701.000000
mean       145.128202
std        760.632845
min         -1.333333
25%          0.066667
50%          0.216667
75%         13.283333
max      12899.083333
Name: tempo_ate_pronto, dtype: float64


In [22]:
pedidos_extremos["tempo_espera_coleta"] = (
    pd.to_datetime(pedidos_extremos["order_moment_collected"])
    - pd.to_datetime(pedidos_extremos["order_moment_ready"])
).dt.total_seconds() / 60

print("Tempo entre pedido pronto e coleta:")
print(pedidos_extremos["tempo_espera_coleta"].describe())

C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\550450444.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(pedidos_extremos["order_moment_collected"])
C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\550450444.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  - pd.to_datetime(pedidos_extremos["order_moment_ready"])


Tempo entre pedido pronto e coleta:
count    2465.000000
mean       29.162934
std       127.996836
min         0.016667
25%         1.233333
50%         3.466667
75%        11.283333
max      2050.383333
Name: tempo_espera_coleta, dtype: float64


In [23]:
pedidos_extremos["tempo_transporte"] = (
    pd.to_datetime(pedidos_extremos["order_moment_delivered"])
    - pd.to_datetime(pedidos_extremos["order_moment_delivering"])
).dt.total_seconds() / 60

print("Tempo entre saída para entrega e entrega:")
print(pedidos_extremos["tempo_transporte"].describe())

C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\2689729814.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(pedidos_extremos["order_moment_delivered"])
C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\2689729814.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  - pd.to_datetime(pedidos_extremos["order_moment_delivering"])


Tempo entre saída para entrega e entrega:
count       238.000000
mean      17177.127941
std       18765.897367
min           0.016667
25%        3712.300000
50%        8810.658333
75%       28682.245833
max      154261.100000
Name: tempo_transporte, dtype: float64


In [24]:
indice_maior_transporte = (
    pedidos_extremos["tempo_transporte"].idxmax()
)

print("Índice do registro:", indice_maior_transporte)

print(
    pedidos_extremos.loc[
        indice_maior_transporte,
        [
            "order_id",
            "order_moment_created",
            "order_moment_delivering",
            "order_moment_delivered",
            "order_moment_finished",
            "order_metric_cycle_time",
            "tempo_transporte"
        ]
    ]
)

Índice do registro: 11529
order_id                               69584292
order_moment_created        1/6/2021 4:07:58 PM
order_moment_delivering     1/6/2021 4:24:11 PM
order_moment_delivered     4/23/2021 7:25:17 PM
order_moment_finished      4/23/2021 7:38:26 PM
order_metric_cycle_time               154277.32
tempo_transporte                       154261.1
Name: 11529, dtype: object


In [25]:
etapas = [
    "tempo_ate_pronto",
    "tempo_espera_coleta",
    "tempo_transporte"
]

print("Total de pedidos extremos:", len(pedidos_extremos))

print("\nRegistros disponíveis por etapa:")
print(pedidos_extremos[etapas].notna().sum())

print("\nPedidos com as três etapas disponíveis:")
print(pedidos_extremos[etapas].notna().all(axis=1).sum())

Total de pedidos extremos: 7469

Registros disponíveis por etapa:
tempo_ate_pronto       6701
tempo_espera_coleta    2465
tempo_transporte        238
dtype: int64

Pedidos com as três etapas disponíveis:
212


In [26]:
pedidos_completos = pedidos_extremos.dropna(
    subset=etapas
).copy()

print("Pedidos com as três etapas:", len(pedidos_completos))

print("\nMedianas por etapa (minutos):")
print(pedidos_completos[etapas].median())

print("\nIntervalos negativos por etapa:")
print((pedidos_completos[etapas] < 0).sum())

Pedidos com as três etapas: 212

Medianas por etapa (minutos):
tempo_ate_pronto          12.408333
tempo_espera_coleta        1.775000
tempo_transporte       10855.916667
dtype: float64

Intervalos negativos por etapa:
tempo_ate_pronto       1
tempo_espera_coleta    0
tempo_transporte       0
dtype: int64


### Investigação dos tempos extremos

Foram identificados 7.469 pedidos finalizados com tempo de ciclo
superior a 24 horas.

A investigação dos intervalos entre eventos revelou:

- Aceitação até pedido pronto: 6.701 registros válidos.
- Pedido pronto até coleta: 2.465 registros válidos.
- Saída para entrega até entrega: 238 registros válidos.
- Apenas 212 pedidos possuem os três intervalos disponíveis.

Nos 212 pedidos, a mediana do intervalo entre saída e entrega
foi de aproximadamente 7,54 dias.

Os resultados indicam a presença de registros temporalmente
atípicos e cobertura insuficiente para uma comparação
representativa entre todas as etapas.

**Decisão metodológica:**

- Preservar os dados originais.
- Não excluir automaticamente pedidos com ciclos superiores a 24 horas.
- Sinalizar tempos extremos para investigação.
- Evitar utilizar esses registros sem tratamento em indicadores
  operacionais sensíveis a valores extremos.
- Priorizar indicadores robustos, como a mediana, quando apropriado.

A investigação detalhada será encerrada nesta etapa para
priorizar a construção dos indicadores de desempenho por loja
e das funcionalidades de inteligência artificial.

In [27]:
caminho_lojas = Path("../data/raw/stores.csv")

lojas = pd.read_csv(caminho_lojas)

print("Dimensões da base de lojas:")
print(lojas.shape)

print("\nColunas disponíveis:")
print(lojas.columns.tolist())

print("\nPrimeiros registros:")
display(lojas.head())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 0: invalid continuation byte

In [28]:
caminho_lojas = Path("../data/raw/stores.csv")

lojas = pd.read_csv(caminho_lojas, encoding="latin-1")

print("Dimensões da base de lojas:")
print(lojas.shape)

print("\nColunas disponíveis:")
print(lojas.columns.tolist())

print("\nPrimeiros registros:")
display(lojas.head())

Dimensões da base de lojas:
(951, 7)

Colunas disponíveis:
['store_id', 'hub_id', 'store_name', 'store_segment', 'store_plan_price', 'store_latitude', 'store_longitude']

Primeiros registros:


,store_id,hub_id,store_name,store_segment,store_plan_price,store_latitude,store_longitude
0,3,2,CUMIURI,FOOD,0.0,NaN,NaN
1,6,3,PIMGUCIS DA VIVA,FOOD,0.0,-30.037415,-51.203520
2,8,3,RASMUR S,FOOD,0.0,-30.037415,-51.203520
3,53,8,PAPA SUCIS,FOOD,0.0,-22.921475,-43.234822
4,54,8,VUZPI PAZZIS,FOOD,0.0,-22.921475,-43.234822


In [29]:
print("store_id existe em pedidos?",
      "store_id" in pedidos.columns)

print("store_id existe em lojas?",
      "store_id" in lojas.columns)

print("\nLojas com store_id duplicado:")
print(lojas["store_id"].duplicated().sum())

print("\nPedidos sem store_id:")
print(pedidos["store_id"].isna().sum())

store_id existe em pedidos? True
store_id existe em lojas? True

Lojas com store_id duplicado:
0

Pedidos sem store_id:
0


In [30]:
lojas_pedidos = set(pedidos["store_id"])
lojas_cadastradas = set(lojas["store_id"])

lojas_sem_cadastro = lojas_pedidos - lojas_cadastradas

print("Lojas diferentes nos pedidos:", len(lojas_pedidos))

print("Lojas cadastradas:", len(lojas_cadastradas))

print("Lojas dos pedidos sem cadastro:", len(lojas_sem_cadastro))

print(
    "Pedidos associados a lojas sem cadastro:",
    pedidos["store_id"].isin(lojas_sem_cadastro).sum()
)

Lojas diferentes nos pedidos: 951
Lojas cadastradas: 951
Lojas dos pedidos sem cadastro: 0
Pedidos associados a lojas sem cadastro: 0


In [31]:
pedidos_lojas = pedidos.merge(
    lojas,
    on="store_id",
    how="left",
    validate="many_to_one"
)

print("Pedidos antes da junção:", len(pedidos))
print("Pedidos após a junção:", len(pedidos_lojas))

print("\nPrimeiros registros:")
display(
    pedidos_lojas[
        ["order_id", "store_id", "store_name", "store_segment"]
    ].head()
)

Pedidos antes da junção: 368999
Pedidos após a junção: 368999

Primeiros registros:


,order_id,store_id,store_name,store_segment
0,68405119,3512,GOI SIPIMUSU,FOOD
1,68405123,3512,GOI SIPIMUSU,FOOD
2,68405206,3512,GOI SIPIMUSU,FOOD
3,68405465,3401,FRUEOU,FOOD
4,68406064,3401,FRUEOU,FOOD


In [32]:
pedidos_por_loja = (
    pedidos_lojas
    .groupby(["store_id", "store_name"])
    .agg(total_pedidos=("order_id", "count"))
    .reset_index()
    .sort_values("total_pedidos", ascending=False)
)

print("Quantidade de lojas analisadas:", len(pedidos_por_loja))

print("\nAs 10 lojas com mais pedidos:")
display(pedidos_por_loja.head(10))

Quantidade de lojas analisadas: 951

As 10 lojas com mais pedidos:


,store_id,store_name,total_pedidos
3,53,PAPA SUCIS,14004
133,676,IUMPICA,12327
152,707,IUMPICA,9271
21,149,IUMPICA,8110
32,274,IUMPICA,7920
125,658,IUMPICA,7433
28,236,IUMPICA,7312
33,294,PAPA SUCIS,6960
202,886,RC OUMILEES,6752
36,321,IUMPICA,6568


In [33]:
pedidos_lojas["cancelado"] = (
    pedidos_lojas["order_status"] == "CANCELED"
)

indicadores_lojas = (
    pedidos_lojas
    .groupby(["store_id", "store_name"])
    .agg(
        total_pedidos=("order_id", "count"),
        pedidos_cancelados=("cancelado", "sum")
    )
    .reset_index()
)

indicadores_lojas["taxa_cancelamento"] = (
    indicadores_lojas["pedidos_cancelados"]
    / indicadores_lojas["total_pedidos"]
    * 100
)

print("Indicadores das 10 lojas com mais pedidos:")

display(
    indicadores_lojas
    .sort_values("total_pedidos", ascending=False)
    .head(10)
)

Indicadores das 10 lojas com mais pedidos:


,store_id,store_name,total_pedidos,pedidos_cancelados,taxa_cancelamento
3,53,PAPA SUCIS,14004,101,0.721223
133,676,IUMPICA,12327,50,0.405614
152,707,IUMPICA,9271,81,0.873692
21,149,IUMPICA,8110,97,1.196054
32,274,IUMPICA,7920,55,0.694444
125,658,IUMPICA,7433,72,0.968653
28,236,IUMPICA,7312,71,0.971007
33,294,PAPA SUCIS,6960,76,1.091954
202,886,RC OUMILEES,6752,226,3.347156
36,321,IUMPICA,6568,138,2.101096


In [34]:
lojas_volume_relevante = indicadores_lojas[
    indicadores_lojas["total_pedidos"] >= 100
].copy()

ranking_cancelamento = (
    lojas_volume_relevante
    .sort_values("taxa_cancelamento", ascending=False)
)

print("Lojas com pelo menos 100 pedidos:")
print(len(lojas_volume_relevante))

print("\nAs 10 maiores taxas de cancelamento:")
display(ranking_cancelamento.head(10))

Lojas com pelo menos 100 pedidos:
308

As 10 maiores taxas de cancelamento:


,store_id,store_name,total_pedidos,pedidos_cancelados,taxa_cancelamento
882,4329,MUSPUO SMIRU,1037,1037,100.000000
134,678,MZU PIUI,103,70,67.961165
178,806,IMLAIZA,279,155,55.555556
302,1125,PIZ RUSMIURIMA,132,60,45.454545
173,791,PULGAIM SIOLLU,186,77,41.397849
179,807,ZIMRPI,314,122,38.853503
594,2278,IPA EMUOU,170,65,38.235294
30,261,AR TRAMUO,670,225,33.582090
456,1633,PIRAGUAM,297,99,33.333333
9,84,PALLO MZU GRALA,2350,780,33.191489


In [35]:
taxa_geral = (
    pedidos_lojas["cancelado"].mean() * 100
)

print(
    "Taxa geral de cancelamento:",
    round(taxa_geral, 2),
    "%"
)

print("\nLojas com taxa acima da média geral:")
print(
    (
        lojas_volume_relevante["taxa_cancelamento"]
        > taxa_geral
    ).sum()
)

Taxa geral de cancelamento: 4.6 %

Lojas com taxa acima da média geral:
132


## Indicadores de desempenho por estabelecimento

As bases de pedidos e estabelecimentos foram relacionadas
utilizando a coluna `store_id`.

### Validação do relacionamento

- Pedidos antes do merge: 368.999
- Pedidos após o merge: 368.999
- Estabelecimentos identificados: 951
- Lojas sem correspondência: 0
- Pedidos perdidos ou duplicados: 0

### Indicadores construídos

1. Volume total de pedidos por estabelecimento.
2. Quantidade de pedidos cancelados.
3. Taxa de cancelamento por estabelecimento.

### Resultados principais

- Taxa geral de cancelamento: 4,60%.
- Lojas com pelo menos 100 pedidos: 308.
- Lojas com taxa acima da referência geral: 132 (42,86%).
- Loja 4329: 1.037 pedidos, todos cancelados.

### Decisões metodológicas

- Utilizar `store_id` como identificador principal.
- Não assumir que estabelecimentos com nomes iguais
  pertencem à mesma rede.
- Aplicar um filtro exploratório de pelo menos 100 pedidos
  para analisar taxas de cancelamento por loja.
- Não interpretar cancelamentos elevados como responsabilidade
  exclusiva dos restaurantes.
- Preservar os dados originais e documentar possíveis anomalias.

### Aplicação no AI Customer Intelligence

Os indicadores poderão alimentar um agente de IA capaz de
identificar estabelecimentos com taxas elevadas de cancelamento,
compará-los com referências da plataforma e sugerir
investigações e ações fundamentadas nos dados.

In [36]:
tempo_por_loja = (
    pedidos_lojas[
        pedidos_lojas["order_status"] == "FINISHED"
    ]
    .groupby("store_id")
    .agg(
        tempo_mediano_ciclo=(
            "order_metric_cycle_time", "median"
        ),
        pedidos_com_tempo=(
            "order_metric_cycle_time", "count"
        )
    )
    .reset_index()
)

print("Lojas com indicador de tempo:")
print(tempo_por_loja["tempo_mediano_ciclo"].notna().sum())

print("\nResumo dos tempos medianos por loja:")
print(tempo_por_loja["tempo_mediano_ciclo"].describe())

Lojas com indicador de tempo:
883

Resumo dos tempos medianos por loja:
count      883.000000
mean       251.829813
std        902.450219
min         13.750000
25%         37.935000
50%         46.585000
75%         86.002500
max      12478.010000
Name: tempo_mediano_ciclo, dtype: float64


In [37]:
ranking_tempos = (
    tempo_por_loja
    .merge(
        lojas[["store_id", "store_name"]],
        on="store_id",
        how="left"
    )
    .sort_values(
        "tempo_mediano_ciclo",
        ascending=False
    )
)

display(ranking_tempos.head(10))

,store_id,tempo_mediano_ciclo,pedidos_com_tempo,store_name
436,1679,12478.010,2,CUMOIARAS
550,2270,9632.500,2,LIGGAPLUS
670,3284,9355.175,2,MZU PIEO SZIP
91,606,8681.270,3,MAP MIP
506,2052,8233.280,1,AERUIR
298,1199,6815.850,1,CISI RUVAVI
658,3128,4385.525,4,GAMGS
232,1014,4087.185,2,OUMGA
347,1378,4008.200,1,SSIRIVSGA
401,1583,3727.330,2,LUVAES


In [38]:
desempenho_lojas = indicadores_lojas.merge(
    tempo_por_loja,
    on="store_id",
    how="left",
    validate="one_to_one"
)

print("Estabelecimentos na tabela consolidada:")
print(len(desempenho_lojas))

print("\nPrimeiros registros:")
display(desempenho_lojas.head())

Estabelecimentos na tabela consolidada:
951

Primeiros registros:


,store_id,store_name,total_pedidos,pedidos_cancelados,taxa_cancelamento,tempo_mediano_ciclo,pedidos_com_tempo
0,3,CUMIURI,13,13,100.000000,NaN,NaN
1,6,PIMGUCIS DA VIVA,3739,88,2.353570,32.18,3649.0
2,8,RASMUR S,2472,172,6.957929,36.08,2299.0
3,53,PAPA SUCIS,14004,101,0.721223,42.70,13895.0
4,54,VUZPI PAZZIS,89,4,4.494382,51.82,85.0


In [39]:
print("Lojas duplicadas:")
print(desempenho_lojas["store_id"].duplicated().sum())

print("\nTotal de pedidos na tabela:")
print(desempenho_lojas["total_pedidos"].sum())

print("\nTotal de cancelamentos:")
print(desempenho_lojas["pedidos_cancelados"].sum())

print("\nLojas sem indicador de tempo:")
print(desempenho_lojas["tempo_mediano_ciclo"].isna().sum())

Lojas duplicadas:
0

Total de pedidos na tabela:
368999

Total de cancelamentos:
16979

Lojas sem indicador de tempo:
68


In [40]:
caminho_saida = Path("../data/processed/store_performance.csv")

desempenho_lojas.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8"
)

print("Arquivo salvo em:")
print(caminho_saida.resolve())

print("\nDimensões da tabela:")
print(desempenho_lojas.shape)

Arquivo salvo em:
C:\Users\Dell\Documents\rocketseat\ai-customer-intelligence\data\processed\store_performance.csv

Dimensões da tabela:
(951, 7)


## Consolidação dos indicadores por estabelecimento

Foi criada a tabela `desempenho_lojas`, reunindo indicadores
operacionais de 951 estabelecimentos.

### Indicadores disponíveis

- `store_id`: identificador único do estabelecimento.
- `store_name`: nome do estabelecimento.
- `total_pedidos`: quantidade total de pedidos.
- `pedidos_cancelados`: quantidade de pedidos cancelados.
- `taxa_cancelamento`: percentual de pedidos cancelados.
- `tempo_mediano_ciclo`: mediana do tempo de ciclo dos pedidos finalizados, em minutos.
- `pedidos_com_tempo`: quantidade de pedidos finalizados com tempo de ciclo disponível.

### Validação

- 951 estabelecimentos preservados.
- Nenhum estabelecimento duplicado.
- 368.999 pedidos contabilizados.
- 16.979 cancelamentos contabilizados.
- 68 estabelecimentos sem indicador de tempo.

### Cuidados na interpretação

O tempo mediano deve ser interpretado considerando a quantidade
de pedidos disponíveis para cada estabelecimento.

Valores ausentes de tempo foram preservados, sem substituição por zero.

A taxa de cancelamento não determina, isoladamente, a causa
dos cancelamentos ou a responsabilidade do estabelecimento.

### Arquivo gerado

`data/processed/store_performance.csv`

A tabela será utilizada nas próximas etapas do projeto,
incluindo consultas SQL, visualização de indicadores e
recomendações assistidas por inteligência artificial.

In [41]:
import sqlite3

caminho_banco = Path(
    "../data/processed/customer_intelligence.db"
)

conexao = sqlite3.connect(caminho_banco)

desempenho_lojas.to_sql(
    "store_performance",
    conexao,
    if_exists="append",
    index=False
)

conexao.commit()
conexao.close()

print("Dados importados para o SQLite!")

Dados importados para o SQLite!
